In [ ]:
import pickle
import os
import pandas as pd
from sklearn.metrics import roc_auc_score
import numpy as np
import sys


In [ ]:
# Ensure all values are numbers, exclude None and lists
def to_number(val):
    if val is None or val == "" or isinstance(val, list):
        return None
    try:
        num = float(val)
        if np.isnan(num):
            return None
        return num
    except (ValueError, TypeError):
        return None

def compute_auroc(accuracies_raw, confidences_raw) -> float:
    accuracies = [to_number(acc) for acc in accuracies_raw]
    confidences = [to_number(conf) for conf in confidences_raw]

    valid_pairs = [
        (acc, conf)
        for acc, conf in zip(accuracies, confidences)
        if acc is not None and conf is not None
    ]

    if len(valid_pairs) == 0:
        return float("nan")

    accuracies_arr = np.array([pair[0] for pair in valid_pairs], dtype=float)
    confidences_arr = np.array([pair[1] for pair in valid_pairs], dtype=float)

    if len(confidences_arr) == 0 or len(np.unique(accuracies_arr)) < 2:
        return float("nan")

    return float(roc_auc_score(accuracies_arr, confidences_arr))

In [ ]:
# load pickled extracted_output
path = "results/trivia_qa/lnll/meta-llama/Meta-Llama-3-8B-Instruct/2025-12-16_01-43-22"  # replace with your actual run directory
path = "results/trivia_qa/lnll/openai/gpt-oss-20b/2025-12-16_02-03-28"  # replace with your actual run directory
# path = "results/trivia_qa/lnll/Qwen/Qwen2.5-7B-Instruct/2025-12-16_01-42-19"  # replace with your actual run directory
path = "results/truthful_qa/lnll/openai/gpt-oss-20b/2025-12-16_02-17-48"
with open(f"{path}/post_filter_model_outputs.pkl", "rb") as f:
    extracted_output = pickle.load(f)

In [ ]:
# load pickled extracted_output
with open(f"{path}/organised_output.pkl", "rb") as f:
    organised_output = pickle.load(f)

In [ ]:
def lnll(logprobs):
    return np.mean(logprobs)
lnll_aurocs = []
for sampling_round in range(10):
    try:
        df = pd.DataFrame({"logprobs": extracted_output[sampling_round].output_logprobs ,"accuracies": organised_output.accuracy_scores[sampling_round]})
        df["logprobs"] = df.logprobs.apply(lambda x: lnll(x) if isinstance(x, list) else x)
        lnll_aurocs.append(compute_auroc(df.accuracies, df.logprobs))
    except Exception as e:
        pass

np.mean(lnll_aurocs)

np.float64(0.5850899766232527)

In [ ]:
def top_p_sum(response_top_k):
    response_top_k_sum = []
    for tk in response_top_k:
        prob_sum = float(np.sum(np.exp([p for _, p in tk])))
        response_top_k_sum.append(prob_sum)
    return -np.std(response_top_k_sum)


topk_normalized_lnll_aurocs = []
for sampling_round in range(1):
    try:
        logprobs = extracted_output[sampling_round].top_k_tokens
        acc = organised_output.accuracy_scores[sampling_round]
        conf = [top_p_sum(lp) for lp in logprobs] 
        topk_normalized_lnll_aurocs.append(compute_auroc(acc, conf))
    except Exception as e:
        print(e)
np.mean(topk_normalized_lnll_aurocs)

np.float64(0.5387556482446993)